In [12]:
import json
import numpy as np
import pandas as pd

# Define explicit ordinal mappings for categorical survey strings
SURVEY_MAPPINGS = {
    'd1_freq_5': {'Never': 0, 'Rarely (1-2 times a week)': 1, 'Sometimes (3-4 times a week)': 2, 'Often (5-6 times a week)': 3, 'Every night': 4, 'Every day': 4},
    'd1_hours': {'Less than 4 hours': 2, '4-5 hours': 4.5, '6-7 hours': 6.5, '7-8 hours': 7.5, 'More than 8 hours': 9},
    'd1_quality': {'Very poor': 0, 'Poor': 1, 'Average': 2, 'Good': 3, 'Very good': 4},
    'd1_freq_3': {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3, 'Always': 4},
    'd1_skipping': {'Never': 0, 'Rarely (1-2 times a month)': 1, 'Sometimes (1-2 times a week)': 2, 'Often (3-4 times a week)': 3, 'Always': 4},
    'd1_impact': {'No impact': 0, 'Minor impact': 1, 'Moderate impact': 2, 'Major impact': 3, 'Severe impact': 4},
    'd1_stress': {'No stress': 0, 'Low stress': 1, 'High stress': 2, 'Extremely high stress': 3},
    'd1_grades': {'Poor': 0, 'Below Average': 1, 'Average': 2, 'Good': 3, 'Excellent': 4},
    'd1_year': {'First year': 1, 'Second year': 2, 'Third year': 3, 'Graduate student': 4},
    
    'd4_bool': {'No': 0, 'Yes': 1},
    'd4_quality': {'Worst ': 0, 'Poor': 1, 'Fair': 2, 'Good': 3, 'Excellent': 4},
    'd4_naps': {'Never': 0, 'Rarely': 1, 'Occasionally': 2, 'Yes, regularly': 3},
    'd4_lights': {'No, I prefer total darkness': 0, 'I use a night light': 1, 'Yes': 2},
    'd4_exercise': {'Never': 0, 'Occasionally': 1, 'A few times a week': 2, 'Yes, daily': 3}
}

def matrix_to_tidy_list(matrix_df):
    """Transforms a square pandas correlation matrix into an array of coordinate objects."""
    tidy_records = []
    columns = list(matrix_df.columns)
    for i, row_name in enumerate(columns):
        for j, col_name in enumerate(columns):
            val = matrix_df.iloc[i, j]
            # Replace NaNs or invalid floats safely for JSON standards
            safe_val = 0.0 if pd.isna(val) or np.isinf(val) else float(val)
            tidy_records.append({
                "row": str(row_name),
                "col": str(col_name),
                "value": round(safe_val, 4)
            })
    return tidy_records

compiled_payload = {}

# --- DATASET 1: Insomnia Outcomes ---
try:
    df1 = pd.read_csv("../dataset/insomnia.csv")
    df1_mapped = pd.DataFrame()
    df1_mapped['Year_of_Study'] = df1.iloc[:, 1].map(SURVEY_MAPPINGS['d1_year'])
    df1_mapped['Sleep_Difficulty_Night'] = df1.iloc[:, 3].map(SURVEY_MAPPINGS['d1_freq_5'])
    df1_mapped['Sleep_Hours_Average'] = df1.iloc[:, 4].map(SURVEY_MAPPINGS['d1_hours'])
    df1_mapped['Wake_Up_Trouble'] = df1.iloc[:, 5].map(SURVEY_MAPPINGS['d1_freq_5'])
    df1_mapped['Sleep_Quality_Rating'] = df1.iloc[:, 6].map(SURVEY_MAPPINGS['d1_quality'])
    df1_mapped['Concentration_Difficulty'] = df1.iloc[:, 7].map(SURVEY_MAPPINGS['d1_freq_3'])
    df1_mapped['Daytime_Fatigue'] = df1.iloc[:, 8].map(SURVEY_MAPPINGS['d1_freq_3'])
    df1_mapped['Miss_Skip_Classes'] = df1.iloc[:, 9].map(SURVEY_MAPPINGS['d1_skipping'])
    df1_mapped['Insufficient_Sleep_Impact'] = df1.iloc[:, 10].map(SURVEY_MAPPINGS['d1_impact'])
    df1_mapped['Device_Before_Bed'] = df1.iloc[:, 11].map(SURVEY_MAPPINGS['d1_freq_5'])
    df1_mapped['Caffeine_Consumption'] = df1.iloc[:, 12].map(SURVEY_MAPPINGS['d1_freq_5'])
    df1_mapped['Physical_Activity'] = df1.iloc[:, 13].map(SURVEY_MAPPINGS['d1_freq_5'])
    df1_mapped['Academic_Stress_Level'] = df1.iloc[:, 14].map(SURVEY_MAPPINGS['d1_stress'])
    df1_mapped['Academic_Performance'] = df1.iloc[:, 15].map(SURVEY_MAPPINGS['d1_grades'])
    compiled_payload["dataset1"] = matrix_to_tidy_list(df1_mapped.dropna().corr(method='pearson'))
except Exception as e:
    print(f"Skipping Dataset 1 due to error: {e}")

# --- DATASET 2: CMU Sensor Wearables ---
try:
    df2 = pd.read_csv("../dataset/cmu.csv")
    for col in ['term_units', 'Zterm_units_ZofZ']:
        df2[col] = pd.to_numeric(df2[col], errors='coerce')
    df2_numeric = df2.copy()
    for col in ['cohort', 'demo_race', 'demo_gender', 'demo_firstgen']:
        df2_numeric[col] = df2_numeric[col].astype('category').cat.codes
    df2_numeric = df2_numeric.drop(columns=['subject_id', 'study'], errors='ignore')
    compiled_payload["dataset2"] = matrix_to_tidy_list(df2_numeric.dropna().corr(method='pearson'))
except Exception as e:
    print(f"Skipping Dataset 2 due to error: {e}")

# --- DATASET 3: Student Sleep Patterns ---
try:
    df3 = pd.read_csv("../dataset/student_sleep_patterns.csv")
    df3_numeric = df3.copy()
    df3_numeric['Gender'] = df3_numeric['Gender'].astype('category').cat.codes
    df3_numeric['University_Year'] = df3_numeric['University_Year'].astype('category').cat.codes
    df3_numeric = df3_numeric.drop(columns=['Student_ID'], errors='ignore')
    compiled_payload["dataset3"] = matrix_to_tidy_list(df3_numeric.dropna().corr(method='pearson'))
except Exception as e:
    print(f"Skipping Dataset 3 due to error: {e}")

# --- DATASET 4: Bedtime Routine Survey ---
try:
    df4 = pd.read_csv("../dataset/sleep_habits.csv")
    df4_mapped = pd.DataFrame()
    df4_mapped['Difficulty_Falling_Staying_Asleep'] = df4.iloc[:, 5].map(SURVEY_MAPPINGS['d4_bool'])
    df4_mapped['Sleep_Quality_Rating'] = df4.iloc[:, 7].map(SURVEY_MAPPINGS['d4_quality'])
    df4_mapped['Daytime_Naps'] = df4.iloc[:, 9].map(SURVEY_MAPPINGS['d4_naps'])
    df4_mapped['Lights_On_During_Sleep'] = df4.iloc[:, 10].map(SURVEY_MAPPINGS['d4_lights'])
    df4_mapped['Exercise_Regularity'] = df4.iloc[:, 11].map(SURVEY_MAPPINGS['d4_exercise'])
    activities = df4.iloc[:, 4].fillna('').astype(str)
    df4_mapped['Bedtime_Act_Electronics'] = activities.str.contains('electronic', case=False).astype(int)
    df4_mapped['Bedtime_Act_TV_Movies'] = activities.str.contains('TV|movies', case=False).astype(int)
    df4_mapped['Bedtime_Act_Reading'] = activities.str.contains('reading', case=False).astype(int)
    compiled_payload["dataset4"] = matrix_to_tidy_list(df4_mapped.dropna().corr(method='pearson'))
except Exception as e:
    print(f"Skipping Dataset 4 due to error: {e}")

# --- DATASET 5: Mental Health Metrics ---
try:
    df5 = pd.read_csv("../dataset/mental_health.csv")
    df5_numeric = df5.copy()
    df5_numeric['gender'] = df5_numeric['gender'].astype('category').cat.codes
    df5_numeric['platform'] = df5_numeric['platform'].astype('category').cat.codes
    df5_numeric['mental_state'] = df5_numeric['mental_state'].astype('category').cat.codes
    df5_numeric = df5_numeric.drop(columns=['person_name', 'date'], errors='ignore')
    compiled_payload["dataset5"] = matrix_to_tidy_list(df5_numeric.dropna().corr(method='pearson'))
except Exception as e:
    print(f"Skipping Dataset 5 due to error: {e}")

# Write as a globally accessible JavaScript object module
output_filename = "data_manifest.js"
with open(output_filename, "w", encoding="utf-8") as file_out:
    file_out.write(f"const correlationData = {json.dumps(compiled_payload, indent=2)};\n")

print(f"Success! Generated clean visual manifest file at: {output_filename}")

Success! Generated clean visual manifest file at: data_manifest.js
